# CharXiv Baseline — Checkpoint 1

**Question.** For each of **3 target vision-language models** — `GPT-4o`, `Claude-3-5-Sonnet`, and
`GPT-4o-Random` — predict whether it answers a CharXiv **reasoning** item correctly. We fit a
**separate classifier per target model**, so every result below is reported per target.

This is the Checkpoint-1 *back-of-the-envelope* learnability test:
- **Baseline features = item metadata** (`category`, `inst_category`, `year`). This is the cheap
  "what kind of item is it" reference the engineered features must beat.
- **The 12 DeLeAn demand dimensions are ENGINEERED features and are excluded from these baselines** —
  their added value is tested in Checkpoint 2. The scientific question becomes: *do the DeLeAn dims
  beat a cheap metadata-only baseline?*
- `GPT-4o-Random` is a random-answer control that fails about 90 percent of items. It is the negative
  control for the reasoning-demand signal: the cognitive demand of an item cannot predict a random
  answer, so the demand dimensions should add little for it. A random guess's chance of being correct
  does, however, depend on the answer format, which the item metadata partly encodes, so metadata
  alone can still rank the control's items.

**Conventions:** positive class = **failure** (`y=1` = incorrect); fixed 800/200 split
(seed `20260618`); paper-grouped CV (`StratifiedGroupKFold`, 5×3, by `paperid`); the 200 test items
are never read. Primary KPI **ROC-AUC** (see `kpis.md`).

## 1. Setup and data (TRAIN only)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.features.preprocessing as P

CAT_COLS = P.CATEGORICAL            # item metadata (baseline categoricals)
TARGETS = P.TARGET_MODELS           # ['GPT-4o', 'Claude-3-5-Sonnet', 'GPT-4o-Random']

con = P.connect()
train_ids, test_ids = P.get_split(con)          # 800 / 200, cached
df = P.make_design_matrix(con, train_ids)        # one row per item, with a y__<model> label per target
con.close()

print(f"train items: {len(df)} | test items (sealed): {len(test_ids)} | targets: {TARGETS}")

train items: 800 | test items (sealed): 200 | targets: ['GPT-4o', 'Claude-3-5-Sonnet', 'GPT-4o-Random']


### The design matrix\n\n`y__<model> = 1` marks **failure** (that target model got the item wrong). Baseline features are the item metadata only; the 12 demand dimensions wait until Checkpoint 2.

In [2]:
print("rows:", df.shape[0], "| feature columns (baseline):", CAT_COLS)
print("failure rate by target model:")
display(df[P.TARGET_YCOLS].mean().rename("failure_rate").round(3).to_frame())

rows: 800 | feature columns (baseline): ['category', 'inst_category', 'year']
failure rate by target model:


,failure_rate
y__GPT-4o,0.514
y__Claude-3-5-Sonnet,0.390
y__GPT-4o-Random,0.899


## 2. Data assessment

In [3]:
# Class balance (failure rate) per target
bal = df[P.TARGET_YCOLS].agg(["mean"]).T
bal.columns = ["failure_rate"]
bal["solve_rate"] = 1 - bal["failure_rate"]
bal.index = TARGETS
print("GPT-4o and Claude-3-5-Sonnet are close to balanced; GPT-4o-Random almost always fails.")
display(bal.round(3))

GPT-4o and Claude-3-5-Sonnet are close to balanced; GPT-4o-Random almost always fails.


,failure_rate,solve_rate
GPT-4o,0.514,0.486
Claude-3-5-Sonnet,0.390,0.610
GPT-4o-Random,0.899,0.101


In [4]:
# Metadata coverage and missingness
print("Missing values in the baseline design matrix:",
      int(df[CAT_COLS + P.TARGET_YCOLS].isna().sum().sum()), "(complete)")
for col in CAT_COLS:
    vc = df[col].value_counts().sort_index()
    print(f"{col} ({vc.size} levels): " + ", ".join(f"{k}:{v}" for k, v in vc.items()))

Missing values in the baseline design matrix: 0 (complete)
category (8 levels): cs:105, econ:109, eess:89, math:108, physics:104, q-bio:97, q-fin:97, stat:91
inst_category (4 levels): 1:356, 2:77, 3:186, 4:181
year (4 levels): 20:197, 21:205, 22:194, 23:204


In [5]:
# Leakage / validity notes
train_papers = set(df["paperid"])
test_meta = pd.read_sql_query("SELECT item_id, paperid FROM item", P.connect()).set_index("item_id")
test_papers = {test_meta.at[i, "paperid"] for i in test_ids}
straddle = len(train_papers & test_papers)
print("- Features are item-intrinsic metadata; no model's correctness (including Human) is an input.")
print(f"- {straddle} papers straddle the item-level 800/200 boundary (known small leak); CV below is "
      "paper-grouped so no paper straddles a fold.")
print("- The 200 test items are never read here.")

- Features are item-intrinsic metadata; no model's correctness (including Human) is an input.
- 6 papers straddle the item-level 800/200 boundary (known small leak); CV below is paper-grouped so no paper straddles a fold.
- The 200 test items are never read here.


## 3. Back-of-the-envelope baselines

Grouped CV (`StratifiedGroupKFold`, 5×3, by `paperid`), fit **separately for each target model**.
Trivial (`DummyClassifier`) → linear (`LogisticRegression`, scaled) → tree (`RandomForest`).
**Positive class = failure.** **No 12 DeLeAn dims** — item metadata only.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
METRICS = ["roc_auc", "bal_acc", "f1"]

def pre(scale):
    ct = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                             CAT_COLS)], remainder="drop")
    steps = [("ct", ct)] + ([("sc", StandardScaler(with_mean=False))] if scale else [])
    return Pipeline(steps)

def learners():
    return {
        "Dummy (most_frequent)": lambda: Pipeline([("pre", pre(False)),
                                                   ("clf", DummyClassifier(strategy="most_frequent"))]),
        "Dummy (stratified)":    lambda: Pipeline([("pre", pre(False)),
                                                   ("clf", DummyClassifier(strategy="stratified", random_state=0))]),
        "LogisticRegression":    lambda: Pipeline([("pre", pre(True)),
                                                   ("clf", LogisticRegression(max_iter=1000))]),
        "RandomForest":          lambda: Pipeline([("pre", pre(False)),
                                                   ("clf", RandomForestClassifier(
                                                       n_estimators=200, min_samples_leaf=5,
                                                       max_features="sqrt", random_state=0, n_jobs=1))]),
    }

def eval_cv(make_est, X, y, groups, n_splits=5, n_repeats=3):
    rows = []
    for r in range(n_repeats):
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=r).split(X, y, groups):
            assert not (set(groups[tr]) & set(groups[te])), "paper straddles a fold!"
            est = make_est().fit(X.iloc[tr], y[tr])
            p = est.predict_proba(X.iloc[te])[:, 1]; pred = est.predict(X.iloc[te])
            rows.append(dict(roc_auc=roc_auc_score(y[te], p),
                             bal_acc=balanced_accuracy_score(y[te], pred),
                             f1=f1_score(y[te], pred, zero_division=0)))
    return pd.DataFrame(rows)

def run_target(target):
    X = df[CAT_COLS]; y = df[P.ycol(target)].to_numpy(); g = df["paperid"].to_numpy()
    out = []
    for name, mk in learners().items():
        s = eval_cv(mk, X, y, g)
        out.append({"target": target, "learner": name,
                    **{c: round(float(s[c].mean()), 4) for c in METRICS},
                    "roc_auc_sd": round(float(s["roc_auc"].std(ddof=0)), 4)})
    return pd.DataFrame(out)

In [7]:
kpi = pd.concat([run_target(t) for t in TARGETS], ignore_index=True)
display(kpi.set_index(["target", "learner"]))

roc_auc  bal_acc      f1  roc_auc_sd
target            learner                                                    
GPT-4o            Dummy (most_frequent)   0.5000   0.5000  0.6788      0.0000
                  Dummy (stratified)      0.5256   0.5256  0.5130      0.0418
                  LogisticRegression      0.5282   0.5107  0.5269      0.0470
                  RandomForest            0.5044   0.4969  0.5143      0.0367
Claude-3-5-Sonnet Dummy (most_frequent)   0.5000   0.5000  0.0000      0.0000
                  Dummy (stratified)      0.4936   0.4936  0.3872      0.0365
                  LogisticRegression      0.5360   0.5324  0.2712      0.0375
                  RandomForest            0.5196   0.5142  0.2573      0.0312
GPT-4o-Random     Dummy (most_frequent)   0.5000   0.5000  0.9467      0.0000
                  Dummy (stratified)      0.5090   0.5090  0.9073      0.0276
                  LogisticRegression      0.7248   0.5156  0.9436      0.0549
                  RandomForest            0.7469   0.4984  0.9451      0.0560

## 4. KPI table

In [8]:
cols = ["target", "learner", "roc_auc", "roc_auc_sd", "bal_acc", "f1"]
kpi_table = kpi[cols].copy()
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
kpi_table.to_csv(RESULTS / "11_charxiv_baseline_kpis.csv", index=False)
print("saved -> results/11_charxiv_baseline_kpis.csv")
display(kpi_table.set_index(["target", "learner"]).round(4))

saved -> results/11_charxiv_baseline_kpis.csv


roc_auc  roc_auc_sd  bal_acc      f1
target            learner                                                    
GPT-4o            Dummy (most_frequent)   0.5000      0.0000   0.5000  0.6788
                  Dummy (stratified)      0.5256      0.0418   0.5256  0.5130
                  LogisticRegression      0.5282      0.0470   0.5107  0.5269
                  RandomForest            0.5044      0.0367   0.4969  0.5143
Claude-3-5-Sonnet Dummy (most_frequent)   0.5000      0.0000   0.5000  0.0000
                  Dummy (stratified)      0.4936      0.0365   0.4936  0.3872
                  LogisticRegression      0.5360      0.0375   0.5324  0.2712
                  RandomForest            0.5196      0.0312   0.5142  0.2573
GPT-4o-Random     Dummy (most_frequent)   0.5000      0.0000   0.5000  0.9467
                  Dummy (stratified)      0.5090      0.0276   0.5090  0.9073
                  LogisticRegression      0.7248      0.0549   0.5156  0.9436
                  RandomForest            0.7469      0.0560   0.4984  0.9451

## 5. Conclusion

- **Metadata is a weak baseline for the two real models.** With only item metadata (no DeLeAn dims),
  the `LogisticRegression` and `RandomForest` baselines barely edge above the Dummy floor
  (ROC-AUC 0.50) for `GPT-4o` and `Claude-3-5-Sonnet`. Metadata alone says little about whether a real
  model gets a chart right, which sets a low bar for the engineered features to clear.
- **`GPT-4o-Random` behaves differently, and that is informative.** Metadata predicts the random
  control noticeably above chance, because a random guess's chance of being right depends on the
  answer format, which the metadata (especially `inst_category`) partly encodes. This is
  answer-guessability, not reasoning: Checkpoint 2 will show the reasoning-demand dimensions add
  little for the control even though they lift the two real models.
- This sets the **bar the DeLeAn dimensions must clear.** The 12 dims are the *engineered* features:
  **Checkpoint 2** tests whether adding them lifts ROC-AUC over this metadata-only baseline for the
  real models, keeping the paper-grouped split discipline and failure-as-positive convention.
- ROC-AUC stays the primary KPI because it is threshold-free and robust to class balance, with
  balanced accuracy and F1 as secondary readouts.

The 200-item test set remains sealed for the final Checkpoint-5 evaluation.